# 09 · Errors and Exceptions

After this notebook you can read any traceback, write `try / except / else / finally` that does exactly what you mean, design your own exception classes, and build the two error-handling pieces every LLM app needs: a parser that survives messy model JSON, and a retry helper that only retries what a retry can fix.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min · **Needs:** [08 · Classes, OOP and dataclasses](08_classes_oop_dataclasses.ipynb) (custom exceptions are classes)

## Why this matters in interviews

- LLM apps fail at the edges all day: malformed JSON, 429 rate limits, timeouts, tools that throw. Interviewers want to see failure handled on purpose, not hidden with `except: pass`.
- "Which errors would you retry?" is a standard follow-up in every system-design round. The good answer is a classification, not a blanket retry.
- Python rounds love the flow questions: which block runs when, `raise ... from`, why `except Exception` is not `except BaseException`.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `pr07` | How do you get reliable JSON out of a model? |
| `ag19` | How do you handle a tool that fails? |
| `po16` | How do you design a retry policy that does not make things worse? |
| `po07` | You are getting 429s from your model provider — what is the handling? |

## 1. What an exception is

**In plain English:** an exception is Python saying "I cannot carry on with this line". It stops the current function and travels *up* the chain of callers until some `except` block handles it. If nobody does, the program stops and prints a traceback.

**Predict the output** before you run the cell: which of the four strings can `int()` convert?

In [ ]:
def parse_score(text):
    return int(text)

for raw in ["42", " 7 ", "4.5", "seven"]:
    try:
        print(f"{raw!r:>9} -> {parse_score(raw)}")
    except ValueError as e:                     # `as e` gives you the exception object
        print(f"{raw!r:>9} -> {type(e).__name__}: {e}")

`int()` strips surrounding whitespace, so `" 7 "` works, but it refuses `"4.5"` (use `int(float(x))` if you really mean it). The exception object carries the message; `type(e).__name__` is the class name. The loop kept going because each failure was handled inside the loop.

## 2. `try / except / else / finally` — which block runs when

| What happens inside `try` | `except ValueError` | `else` | `finally` | Afterwards |
|---|---|---|---|---|
| nothing goes wrong | skipped | **runs** | **runs** | carries on |
| a `ValueError` is raised | **runs** | skipped | **runs** | carries on |
| a different error (`KeyError`) | skipped | skipped | **runs** | the error keeps travelling up |
| `return` inside `try` | skipped | skipped | **runs** (before the return completes) | function returns |

- `else` is for code that should run only if `try` succeeded. It keeps the `try` block tiny, so your `except` cannot accidentally catch errors from lines you never meant to protect.
- `finally` always runs: use it for cleanup (close a file, release a lock, stop a timer).

The cell below records which blocks actually ran in each case and asserts that the table is right.

In [ ]:
def run(scenario):
    ran = []
    def body():
        try:
            ran.append("try")
            if scenario == "ValueError":
                int("not a number")
            elif scenario == "KeyError":
                {}["missing"]
            elif scenario == "return in try":
                return "early"
        except ValueError:
            ran.append("except")
        else:
            ran.append("else")
        finally:
            ran.append("finally")
    try:
        body()
    except KeyError:
        ran.append("(KeyError reached the caller)")
    return ran

expected = {
    "no error":      ["try", "else", "finally"],
    "ValueError":    ["try", "except", "finally"],
    "KeyError":      ["try", "finally", "(KeyError reached the caller)"],
    "return in try": ["try", "finally"],
}
for scenario, want in expected.items():
    got = run(scenario)
    print(f"{scenario:<14} -> {' -> '.join(got)}")
    assert got == want

One trap worth knowing: a `return` inside `finally` silently throws away any exception that was in flight. Python 3.14 added a `SyntaxWarning` for exactly this (PEP 765). Never return from `finally`.

## 3. The exception hierarchy

Every exception is a class, and they form a tree. `except SomeClass` also catches every **subclass**. That is why the order of `except` clauses matters: the first one that matches wins, so put the most specific first.

In [ ]:
import json
import asyncio

def lineage(exc_type):
    """KeyError -> LookupError -> Exception -> BaseException (drop `object`)."""
    return " -> ".join(c.__name__ for c in exc_type.__mro__[:-1])

for exc_type in [KeyError, IndexError, json.JSONDecodeError, UnicodeDecodeError, FileNotFoundError,
                 TimeoutError, ZeroDivisionError, TypeError, RecursionError,
                 KeyboardInterrupt, SystemExit, asyncio.CancelledError]:
    print(lineage(exc_type))

assert issubclass(json.JSONDecodeError, ValueError)          # so `except ValueError` catches bad JSON too
assert not issubclass(KeyboardInterrupt, Exception)           # Ctrl-C escapes `except Exception` on purpose
assert not issubclass(asyncio.CancelledError, Exception)      # so does task cancellation (since Python 3.8)

**`Exception` vs `BaseException`:** almost everything inherits from `Exception`. Three important things do not: `KeyboardInterrupt` (Ctrl-C), `SystemExit` (`sys.exit()`), and `asyncio.CancelledError` (a cancelled task or a timeout). They sit directly under `BaseException` so that `except Exception` does **not** trap them — you can still stop the program or cancel a task. Catch `Exception` at most; never `BaseException` unless you re-raise.

Where you meet the common ones in AI code:

| Exception | Typical cause in an LLM app |
|---|---|
| `KeyError` | the model's JSON lacks a key you index with `data["score"]` |
| `IndexError` | `resp.choices[0]` on an empty list, `hits[0]` when retrieval found nothing |
| `ValueError` / `json.JSONDecodeError` | unparseable model output, `float("N/A")` |
| `TypeError` | `message.content` is `None` because the model returned a tool call instead of text |
| `UnicodeDecodeError` | opening a file without `encoding="utf-8"` on Windows |
| `FileNotFoundError` | a relative path that depends on the current working directory |
| `TimeoutError` / `ConnectionError` | the network — usually worth one retry |
| `RecursionError` | recursing over a deeply nested JSON or tree |

In [ ]:
def classify(exc):
    try:
        raise exc
    except json.JSONDecodeError:            # most specific first
        return "bad JSON"
    except ValueError:
        return "some other ValueError"
    except LookupError:                     # parent of both KeyError and IndexError
        return "missing key or index"

print(classify(json.JSONDecodeError("Expecting value", doc="", pos=0)))
print(classify(ValueError("could not convert string to float: 'N/A'")))
print(classify(IndexError("list index out of range")))
assert classify(KeyError("score")) == "missing key or index"

## 4. Raising exceptions — with a message that helps

`raise SomeError("message")` stops the function. A good message says **what was wrong, what was expected, and what was received**. In an agent this matters twice: the same message may be handed back to the model so it can fix its own tool call ("a good error is a prompt").

Inside an `except` block, a bare `raise` re-raises the current exception with its original traceback — use it when you only want to log or add context and let the error continue.

In [ ]:
def check_temperature(t):
    if not isinstance(t, (int, float)) or isinstance(t, bool):
        raise TypeError(f"temperature must be a number, got {type(t).__name__} {t!r}")
    if not 0 <= t <= 2:
        raise ValueError(f"temperature must be between 0 and 2, got {t}")
    return float(t)

for t in [0.7, 3, "hot"]:
    try:
        print(f"{t!r:>6} -> ok: {check_temperature(t)}")
    except (TypeError, ValueError) as e:        # a tuple catches several types in one clause
        print(f"{t!r:>6} -> {type(e).__name__}: {e}")

## 5. Custom exception classes (with extra fields)

Define your own exceptions when callers need to react differently to different failures. The usual shape:

- **one base class per library or layer** (`LLMError`), so callers can catch "anything from the LLM layer" in one line;
- **subclasses for the cases callers treat differently** (rate limit vs bad output);
- **extra fields** for the data the handler needs (`status`, `retry_after`, the raw model output). Call `super().__init__(message)` so `str(e)` still works.

In [ ]:
RETRYABLE_STATUS = {408, 409, 429, 500, 502, 503, 504}

class LLMError(Exception):
    """Base class for everything our LLM layer raises."""

class LLMAPIError(LLMError):
    def __init__(self, message, *, status, retry_after=None):
        super().__init__(message)
        self.status = status
        self.retry_after = retry_after          # seconds, if the provider sent a Retry-After header

    @property
    def retryable(self):
        return self.status in RETRYABLE_STATUS

class RateLimitError(LLMAPIError):
    def __init__(self, message="rate limit reached", *, retry_after=None):
        super().__init__(message, status=429, retry_after=retry_after)

class LLMOutputError(LLMError):
    """The model answered, but not in a shape we can use. Keeps the raw text for logs and repair prompts."""
    def __init__(self, message, *, raw):
        super().__init__(message)
        self.raw = raw

for err in [RateLimitError(retry_after=2.0), LLMAPIError("invalid value for 'temperature'", status=400)]:
    try:
        raise err
    except LLMError as e:                       # one clause catches the whole family
        print(f"{type(e).__name__:<14} status={e.status} retryable={e.retryable} "
              f"retry_after={e.retry_after} msg={str(e)!r}")

Real SDKs are built exactly like this. The `openai` package has a base `OpenAIError`, status-code subclasses that carry `.status_code` and the HTTP `.response`, and separate classes for network failures:

In [ ]:
import openai

for cls in [openai.RateLimitError, openai.InternalServerError, openai.BadRequestError,
            openai.AuthenticationError, openai.APITimeoutError, openai.APIConnectionError]:
    mro = cls.__mro__
    print(" -> ".join(c.__name__ for c in mro[:mro.index(Exception)]))

assert issubclass(openai.APITimeoutError, openai.APIConnectionError)   # catching connection errors also catches timeouts

## 6. Chaining: `raise ... from`

When you catch a low-level error and raise a clearer one, keep the original as the **cause**: `raise NewError(...) from e`. The traceback then shows both, joined by *"The above exception was the direct cause of the following exception"*. Without `from`, Python still links them (as `__context__`) but the message reads *"During handling of the above exception, another exception occurred"*, which looks like a bug in your handler. `from None` hides the original on purpose.

You can also attach extra context without changing the type with `e.add_note("...")` (Python 3.11+); notes print under the error message.

In [ ]:
import traceback

def load_config(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        err = LLMOutputError(f"config is not valid JSON ({e.msg} at line {e.lineno}, column {e.colno})", raw=text)
        err.add_note("hint: check for a trailing comma")
        raise err from e

try:
    load_config('{"model": "gpt-4.1-mini",}')
except LLMOutputError as e:
    print("error    :", e)
    print("cause    :", type(e.__cause__).__name__, "-", e.__cause__)
    print("notes    :", e.__notes__)
    tb_lines = traceback.format_exception(e)            # the full text Python would print
    print("traceback mentions the cause:", any("direct cause" in line for line in tb_lines))
    assert isinstance(e.__cause__, json.JSONDecodeError)

## 7. EAFP vs LBYL

Two styles for "this might not work":

| | **LBYL** — look before you leap | **EAFP** — easier to ask forgiveness than permission |
|---|---|---|
| Code | `if key in d: return d[key]` | `try: return d[key]` / `except KeyError: ...` |
| Pythonic? | fine for cheap, obvious checks | the idiomatic default in Python |
| Race conditions | yes: the file can vanish between `exists()` and `open()` | no: you just try, and handle failure |
| Cost | a check every time | nearly free when nothing is raised (Python 3.11+ "zero-cost" `try`); raising is the expensive part |

For dictionaries the best answer is often neither: `d.get(key, default)`. The timing below shows the cost model: `try` costs almost nothing when it succeeds, and raising plus catching costs several times more. So EAFP suits the common case succeeding; LBYL suits the common case failing.

In [ ]:
import timeit
import matplotlib.pyplot as plt

BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "axes.grid.axis": "y", "grid.color": "#e1e0d9", "axes.axisbelow": True})

setup = "d = {'model': 'gpt-4.1-mini'}"
snippets = {
    ("LBYL", "key present"): "v = d['model'] if 'model' in d else None",
    ("LBYL", "key missing"): "v = d['temperature'] if 'temperature' in d else None",
    ("EAFP", "key present"): "try:\n    v = d['model']\nexcept KeyError:\n    v = None",
    ("EAFP", "key missing"): "try:\n    v = d['temperature']\nexcept KeyError:\n    v = None",
}

def ns_per_run(stmt, n=200_000):
    return min(timeit.repeat(stmt, setup, number=n, repeat=7)) / n * 1e9

timings = {key: ns_per_run(stmt) for key, stmt in snippets.items()}
for (style, case), ns in timings.items():
    print(f"{style} / {case:<12}: {ns:6.1f} ns")
ratio = timings[("EAFP", "key missing")] / timings[("EAFP", "key present")]
print(f"raising + catching KeyError costs {ratio:.0f}x a successful lookup")
assert ratio > 2                                     # the try is cheap; the raise is the costly part

In [ ]:
cases = ["key present", "key missing"]
x = range(len(cases))
fig, ax = plt.subplots(figsize=(7, 3.6))
for offset, style, color in [(-0.2, "LBYL", BLUE), (0.2, "EAFP", ORANGE)]:
    vals = [timings[(style, c)] for c in cases]
    bars = ax.bar([i + offset for i in x], vals, width=0.38, color=color, label=style)
    ax.bar_label(bars, fmt="%.0f", padding=2, fontsize=9)
ax.set_xticks(list(x), cases)
ax.set_ylabel("nanoseconds per lookup (lower is faster)")
ax.set_title("try/except is cheap until something is actually raised")
ax.legend(frameon=False)
plt.show()

## 8. Why a bare or broad `except` is dangerous

Two different mistakes:

1. **Bare `except:`** catches `BaseException`, so it even swallows `sys.exit()` and Ctrl-C.
2. **`except Exception:` with no logging** hides your own bugs. A typo becomes a `NameError`, the `except` turns it into a default value, and the pipeline quietly produces garbage.

**Predict:** what does `score_all` return? The function has a typo in it.

In [ ]:
import sys

def score(record):
    try:
        return float(record["score"]) * weigth       # typo: `weigth` does not exist -> NameError
    except Exception:
        return 0.0                                   # "just in case" - hides the bug

def score_all(records):
    return [score(r) for r in records]

print("scores:", score_all([{"score": "0.9"}, {"score": "0.4"}]))

def refuses_to_exit():
    try:
        sys.exit(1)
    except:                                          # bare except: catches SystemExit too
        return "swallowed SystemExit - the program refused to stop"

print(refuses_to_exit())

Every score came back `0.0`, and no error was shown. The fix is to catch **only what you expect and can handle**, and let everything else crash loudly:

```python
def score(record):
    try:
        return float(record["score"]) * weight
    except (KeyError, ValueError):       # missing or unparseable score: a data problem we expect
        return 0.0
```

A broad `except Exception` is acceptable in exactly one place: at a **boundary** (the top of a request handler, a worker loop, a tool runner), and only if it **logs the full traceback** and returns an error response or re-raises. Which brings us to reading and logging tracebacks.

## 9. Reading a traceback

Read it **bottom-up**:

1. The **last line** is the exception type and message — what went wrong.
2. The frame just above it is **where** it was raised (file, line, function, and the source line; Python 3.11+ underlines the exact expression with `^^^^`).
3. Walk upwards until you reach the first frame in **your** code — that is usually where the fix goes. Frames inside libraries are rarely the culprit.

In [ ]:
import re

def parse_line(line):
    name, count = line.split("=")
    return name.strip(), int(count)

def load_counts(text):
    return dict(parse_line(line) for line in text.splitlines() if line.strip())

try:
    load_counts("docs = 12\nchunks = 48\nembeddings = forty")
except ValueError:
    text = traceback.format_exc()
    print(re.sub(r'File "[^"]*[\\/]([^"\\/]+)"', r'File "\1"', text))   # keep only file names, for readability

(In a notebook each cell is compiled as a temporary file with a numeric name, so that is the file name you see.)

Bottom line: `ValueError: invalid literal for int() with base 10: ' forty'` — note the leading space, which `split("=")` left behind; `int()` would have accepted it, but not the word. One frame up: `parse_line`, line `return name.strip(), int(count)`. Above that, the generator expression and `load_counts` just show how we got there. You can also inspect the frames in code — handy for tests and error reports:

In [ ]:
try:
    load_counts("embeddings = forty")
except ValueError as e:
    frames = traceback.extract_tb(e.__traceback__)
    for f in frames:
        print(f"{f.name:<12} line {f.lineno}: {f.line}")
    assert frames[-1].name == "parse_line"          # the innermost frame is where it was raised

## 10. `logging.exception` — log the traceback, not just the message

Inside an `except` block, `logger.exception("...")` logs at ERROR level **and attaches the full traceback**. `print(e)` gives you one line with no location; at 3 a.m. you want the traceback. (`logger.error("...", exc_info=True)` does the same thing.) Use `%s` placeholders rather than f-strings so the message is only formatted if it is actually emitted.

In [ ]:
import io
import logging

log_stream = io.StringIO()
log = logging.getLogger("nb09")
log.handlers.clear()
log.addHandler(logging.StreamHandler(log_stream))
log.handlers[0].setFormatter(logging.Formatter("%(levelname)s %(name)s: %(message)s"))
log.setLevel(logging.INFO)
log.propagate = False

try:
    load_counts("embeddings = forty")
except ValueError:
    log.exception("failed to load counts from %s", "counts.txt")

logged = log_stream.getvalue()
lines = logged.splitlines()
print(lines[0])
print("   ...", len(lines) - 2, "traceback lines ...")
print(lines[-1])
assert "Traceback (most recent call last)" in logged

## 11. Exception groups and `except*` (Python 3.11+)

When several things run at once — parallel tool calls, an `asyncio.TaskGroup` — more than one can fail. Python bundles them into an `ExceptionGroup`. `except*` handles each *type* inside the group separately, and — unlike normal `except` — **several `except*` clauses can run** for the same group.

In [ ]:
def run_tools_in_parallel():
    raise ExceptionGroup("3 of 4 tool calls failed", [
        TimeoutError("search_docs timed out after 10s"),
        ValueError("calculator: could not parse '2 +* 3'"),
        TimeoutError("get_weather timed out after 10s"),
    ])

handled = []
try:
    run_tools_in_parallel()
except* TimeoutError as group:
    handled.append("timeouts")
    print("retry these :", [str(e) for e in group.exceptions])
except* ValueError as group:
    handled.append("bad args")
    print("tell the model:", [str(e) for e in group.exceptions])

assert handled == ["timeouts", "bad args"]       # both clauses ran

You will see these for real in [14 · async/await basics](14_async_await_basics.ipynb), where `asyncio.TaskGroup` raises an `ExceptionGroup` when tasks fail.

## 12. Practical A: parse messy LLM JSON robustly

Even when you ask for "JSON only", models wrap it in markdown fences, add a friendly sentence, leave a trailing comma, or get cut off by `max_tokens`. The robust pattern:

1. try the text as-is;
2. fall back through cheap, safe repairs: the inside of a fenced block, the outermost `{...}`, trailing commas removed;
3. check the result is an object with the keys you need;
4. otherwise raise **one clear custom error** that keeps the raw text (for logs and for a repair prompt), chained to the underlying `JSONDecodeError`.

Never `eval()` model output. Best of all, avoid the problem with provider structured outputs — see [13 · Type hints and Pydantic](13_type_hints_and_pydantic.ipynb) — but you will still need this for models and gateways that lack them.

In [ ]:
FENCE = re.compile(r"```(?:json)?\s*(.*?)\s*```", re.DOTALL | re.IGNORECASE)
TRAILING_COMMA = re.compile(r",\s*([}\]])")

def json_candidates(raw):
    """Yield (strategy, text) pairs to try, cheapest and safest first."""
    yield "as-is", raw.strip()
    if m := FENCE.search(raw):
        yield "inside ``` fence", m.group(1)
    start, end = raw.find("{"), raw.rfind("}")
    if 0 <= start < end:
        yield "outermost {...}", raw[start:end + 1]

def parse_llm_json(raw, required=()):
    """Return (object, strategy that worked) or raise LLMOutputError."""
    last_err = None
    for strategy, text in json_candidates(raw):
        for fix, candidate in [("", text), (" + trailing comma removed", TRAILING_COMMA.sub(r"\1", text))]:
            try:
                data = json.loads(candidate)
            except json.JSONDecodeError as e:
                last_err = e
                continue
            if not isinstance(data, dict):
                raise LLMOutputError(f"expected a JSON object, got {type(data).__name__}", raw=raw)
            if missing := [k for k in required if k not in data]:
                raise LLMOutputError(f"JSON object is missing required keys {missing}", raw=raw)
            return data, strategy + fix
    hint = " - it looks truncated; raise max_tokens" if raw.count("{") > raw.count("}") else ""
    raise LLMOutputError(f"no valid JSON object in the model output{hint}", raw=raw) from last_err

In [ ]:
outputs = {
    "clean":          '{"sentiment": "positive", "score": 0.91}',
    "fenced":         '```json\n{"sentiment": "positive", "score": 0.91}\n```',
    "chatty":         'Sure! Here is the JSON:\n{"sentiment": "positive", "score": 0.91}\nAnything else?',
    "trailing comma": '{"sentiment": "positive", "score": 0.91,}',
    "missing key":    '{"sentiment": "positive"}',
    "truncated":      '{"sentiment": "positive", "sco',
    "refusal":        "I'm sorry, I can't help with that.",
}
parsed, failed = {}, {}
for name, raw in outputs.items():
    try:
        data, how = parse_llm_json(raw, required=("sentiment", "score"))
        parsed[name] = data
        print(f"{name:<15} OK    via {how:<35} {data}")
    except LLMOutputError as e:
        failed[name] = e
        print(f"{name:<15} ERROR {e}")

assert all(d == {"sentiment": "positive", "score": 0.91} for d in parsed.values()) and len(parsed) == 4
assert failed["truncated"].raw == outputs["truncated"]                  # raw text kept for logs / repair
assert isinstance(failed["truncated"].__cause__, json.JSONDecodeError)  # chained to the real parse error

Four messy outputs were rescued and three failures produced **specific** errors. In production you would count how often each fallback strategy fires — a rising "outermost {...}" rate is an early warning that a prompt or model change made outputs chattier. Next step for the failures: send `e` back to the model in a repair prompt (notebook 13 does this with pydantic's error messages).

## 13. Practical B: tool failures become observations, not crashes

In an agent loop a tool error should not kill the run. Turn it into a short message the model can act on, and decide whether trying again could help. Bugs in *your* code get logged with a traceback, and the model only sees a generic message (stack traces waste tokens and leak internals).

In [ ]:
def get_order_status(order_id):
    if not re.fullmatch(r"ORD-\d{6}", order_id):
        raise ValueError(f"order_id must look like ORD-123456; got {order_id!r}")
    if order_id == "ORD-999999":
        raise TimeoutError("order service did not answer within 2s")
    return {"order_id": order_id, "status": "SHIPPED"}

def run_tool(fn, **kwargs):
    """Run a tool for an agent. Always returns a JSON string observation for the model."""
    try:
        return json.dumps({"ok": True, "result": fn(**kwargs)})
    except (ValueError, TypeError) as e:        # bad arguments from the model: say how to fix them
        return json.dumps({"ok": False, "error": str(e), "retryable": False})
    except TimeoutError as e:                   # transient: one retry is reasonable
        return json.dumps({"ok": False, "error": str(e), "retryable": True})
    except Exception:                           # a bug in OUR tool: full traceback to the logs only
        log.exception("tool %s crashed with args %s", fn.__name__, kwargs)
        return json.dumps({"ok": False, "error": "internal tool error", "retryable": False})

def buggy_tool(order_id):
    return {"status": {}["status"]}             # a KeyError bug

for fn, kwargs in [(get_order_status, {"order_id": "ORD-123456"}), (get_order_status, {"order_id": "12345"}),
                   (get_order_status, {}), (get_order_status, {"order_id": "ORD-999999"}),
                   (buggy_tool, {"order_id": "ORD-123456"})]:
    print(f"{fn.__name__}({kwargs}) -> {run_tool(fn, **kwargs)}")

error_lines = [line for line in log_stream.getvalue().splitlines() if line.startswith("ERROR")]
print("\nthe log (not the model) got the details:", error_lines[-1])
assert "KeyError" in log_stream.getvalue()      # the traceback is in the log...
assert "KeyError" not in run_tool(buggy_tool, order_id="ORD-123456")   # ...but never in the observation

## 14. Practical C: retry only what a retry can fix

The classification comes first, the mechanics second:

| Retry (something outside may change) | Do **not** retry (it will fail the same way again) |
|---|---|
| timeouts, connection resets | 400 bad request, 422 invalid arguments |
| 429 rate limit (honour `Retry-After`) | 401 / 403 authentication and permission |
| 500, 502, 503, 504, and 408 / 409 | schema-validation failures, content-policy refusals |

Below, `sleep` is a parameter so the demo records the waits instead of actually sleeping — the same trick makes retry code easy to unit-test ([15 · Testing and debugging](15_testing_and_debugging.ipynb)). Exponential backoff *with jitter* is built properly in [12 · Decorators and context managers](12_decorators_and_context_managers.ipynb).

In [ ]:
import time

def is_retryable(exc):
    if isinstance(exc, LLMAPIError):
        return exc.retryable
    return isinstance(exc, (TimeoutError, ConnectionError))

def call_with_retry(fn, *, max_attempts=4, base_delay=0.5, sleep=time.sleep):
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except Exception as e:
            if not is_retryable(e) or attempt == max_attempts:
                raise                                   # terminal error, or out of attempts
            delay = getattr(e, "retry_after", None) or base_delay * 2 ** (attempt - 1)
            sleep(delay)

def scripted(*outcomes):
    """A fake API call that raises or returns the given outcomes in order."""
    queue = list(outcomes)
    def call():
        call.attempts += 1
        outcome = queue.pop(0)
        if isinstance(outcome, BaseException):
            raise outcome
        return outcome
    call.attempts = 0
    return call

In [ ]:
scenarios = {
    "429 then 503 then OK": scripted(RateLimitError(retry_after=2.0), LLMAPIError("overloaded", status=503), "answer"),
    "400 bad request":      scripted(LLMAPIError("invalid value for 'temperature'", status=400)),
    "timeout x4":           scripted(*[TimeoutError("read timed out")] * 4),
}
results = {}
for name, fake_call in scenarios.items():
    waits = []
    try:
        outcome = f"returned {call_with_retry(fake_call, sleep=waits.append)!r}"
    except Exception as e:
        outcome = f"raised {type(e).__name__}: {e}"
    results[name] = (fake_call.attempts, waits)
    print(f"{name:<21} attempts={fake_call.attempts}  waited={waits}  -> {outcome}")

assert results["429 then 503 then OK"] == (3, [2.0, 1.0])     # honoured Retry-After, then backed off
assert results["400 bad request"] == (1, [])                  # never retried a request that cannot succeed
assert results["timeout x4"] == (4, [0.5, 1.0, 2.0])           # gave up after max_attempts

With the real SDK the same classification is one tuple, `(openai.RateLimitError, openai.InternalServerError, openai.APITimeoutError, openai.APIConnectionError)`. One senior detail: the `openai` client **already retries** twice by default (`max_retries=2`) with backoff. Wrap your own retry around it and you multiply the attempts (3 × 4 = 12 calls per request during an outage). Either use the SDK's retries, or create the client with `max_retries=0` and own the policy yourself.

## Try it yourself

**1.** Write `safe_float(text, default=None)` in EAFP style. It should return `default` for `"N/A"`, `""` and `None` (hint: `float(None)` raises `TypeError`, not `ValueError`).

In [ ]:
# Solution — try it yourself first, then run this
def safe_float(text, default=None):
    try:
        return float(text)
    except (TypeError, ValueError):
        return default

assert safe_float("0.25") == 0.25
assert safe_float(" 3 ") == 3.0
assert safe_float("N/A") is None and safe_float("") is None and safe_float(None, default=0.0) == 0.0
print("safe_float passes all checks")

**2.** Create `ToolArgumentError`, a subclass of `ValueError` with fields `tool`, `param` and `got`. Raise it from a `set_top_k(k)` tool when `k` is not between 1 and 50, and show that old code with `except ValueError` still catches it.

In [ ]:
# Solution — try it yourself first, then run this
class ToolArgumentError(ValueError):
    def __init__(self, tool, param, got, expected):
        super().__init__(f"{tool}: {param} must be {expected}; got {got!r}")
        self.tool, self.param, self.got = tool, param, got

def set_top_k(k):
    if not (isinstance(k, int) and 1 <= k <= 50):
        raise ToolArgumentError("set_top_k", "k", k, "an integer from 1 to 50")
    return k

try:
    set_top_k(500)
except ValueError as e:                      # subclassing ValueError keeps old handlers working
    print(f"caught {type(e).__name__}: {e}  (param={e.param}, got={e.got})")
    assert isinstance(e, ToolArgumentError)

**3.** Four parallel calls failed with `[RateLimitError(), LLMAPIError('bad', status=400), TimeoutError(), LLMAPIError('auth', status=401)]`. Use `ExceptionGroup.split` with `is_retryable` to separate what should be retried from what should be reported.

In [ ]:
# Solution — try it yourself first, then run this
group = ExceptionGroup("4 calls failed", [RateLimitError(), LLMAPIError("bad", status=400),
                                          TimeoutError("slow"), LLMAPIError("auth", status=401)])
retry, report = group.split(is_retryable)
print("retry :", [type(e).__name__ for e in retry.exceptions])
print("report:", [f"{type(e).__name__}({e.status})" for e in report.exceptions])
assert len(retry.exceptions) == 2 and len(report.exceptions) == 2

## Say it in an interview

- **30-second answer on flow:** "`try` holds only the line that can fail; `except` handles the specific errors I expect; `else` runs if nothing failed; `finally` always runs, for cleanup. The first matching `except` wins, so specific ones go first."
- **On broad excepts:** "I catch the narrowest exception I can handle. A broad `except Exception` belongs only at a boundary — request handler, worker loop, tool runner — and it logs the traceback with `logger.exception`. Bare `except:` also catches `SystemExit` and `KeyboardInterrupt`, so I never write it."
- **On custom errors:** "One base class per layer, subclasses for cases the caller treats differently, extra fields like `status` and `retry_after`, and `raise ... from e` so the root cause stays in the traceback."
- **On retries (po16 / po07):** "Classify first: retry timeouts, connection errors, 429 and 5xx; never retry 400/401/403/422 or validation failures. Then cap attempts and total time, back off exponentially with jitter, honour `Retry-After`, and use idempotency keys for side effects. The OpenAI SDK already retries twice, so don't stack retries blindly."
- **On LLM JSON (pr07):** "Prefer provider structured outputs. Otherwise parse defensively — strip fences, take the outermost object, fix trailing commas, never `eval` — then validate, and on failure raise a named error and feed the message back to the model once or twice."
- **Traps they probe:** `except A, B:` is Python 2 syntax (use a tuple `except (A, B):`); `return` in `finally` swallows errors; `except Exception` does not catch `KeyboardInterrupt` or `asyncio.CancelledError`, and that is deliberate; `assert` is not input validation (it disappears with `python -O`).

## Next

- [10 · Files, JSON, CSV and pathlib](10_files_json_csv_pathlib.ipynb) — the I/O that raises most of these errors
- [12 · Decorators and context managers](12_decorators_and_context_managers.ipynb) — a retry decorator with backoff and jitter
- [13 · Type hints and Pydantic](13_type_hints_and_pydantic.ipynb) — `ValidationError` and repair prompts
- [14 · async/await basics](14_async_await_basics.ipynb) — `TaskGroup`, cancellation and `ExceptionGroup` in practice
- [16 · Python gotchas interviewers love](16_python_gotchas_interviewers_love.ipynb)
- Production depth: [Retries, backoff, rate limits and circuit breakers](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb) · [Structured outputs with Pydantic](../08_llm_apps/03_structured_outputs_with_pydantic.ipynb) · [Tool design and function schemas](../15_agentic_ai/02_tool_design_and_function_schemas.ipynb)
- Theory: [python_basics course](../../python_basics/index.html) · [interview bank](../../ai_interview_prep/index.html)